# Pipeline

## Imports

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import torch

import resnet as features
import data_utils

TRAIN_CSV = "../../data/tabular/train/train.csv"
TEST_CSV = "../../data/tabular/test/test.csv"
IMAGE_FOLDER = "../../data/image" 

## Load and preprocess data with ResNet50 to extract features

In [ ]:
print("Loading Training Data...")
df_train, y_train = data_utils.load_train_data(TRAIN_CSV)
print(f"   Found {len(df_train)} unique training images.")

print("Extracting Features (ResNet)...")
# Check if we have a GPU, otherwise CPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
extractor = features.ResNetExtractor(device=device)

def get_full_path(p): 
    return os.path.join(IMAGE_FOLDER, p)

# Extract features for training images
# This might take a few minutes depending on your hardware
train_feats = []
for idx, path in enumerate(df_train['image_path']):
    if idx % 50 == 0: print(f"   Processed {idx}/{len(df_train)}...")
    train_feats.append(extractor.get_features(get_full_path(path)))

X_train_raw = np.array(train_feats)

## PCA for dimensionality reduction

In [ ]:
print("Reducing Dimensions (PCA)...")
# Reduce 2048 features -> 50 features
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_raw)

## Prediction with regression forest

In [ ]:
print("4. Training Random Forest (Multi-Output)...")
# Split into internal train/val set to check performance
X_tr, X_val, y_tr, y_val = train_test_split(X_train_raw, y_train, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

# Evaluation
preds_val = rf.predict(X_val)
score = data_utils.weighted_global_r2(y_val, preds_val)
# RMSE for Total Biomass (Index 0 is Dry_Total_g)
rmse = data_utils.rmse_per_target(y_val, preds_val)

print(f"   Validation R^2 Score: {score:.4f}")
print(rmse)